# PSE-823: Advanced Process Dynamics and Control
## Lecture 11 Companion -- Full Derivations and Worked Answers
### (Coughanowr & LeBlanc, Ch. 20)

### Setup (run once)

- Imports used below

In [1]:
import numpy as np
import sympy as sp
from scipy.special import erfc

## Part A
### Kettle: from Eq. 20.1 to Eq. 20.9

(Coughanowr & LeBlanc, Ch. 20, Sec. 20.1)
<!-- source: coughanowr-ch20 -->

### Linearized water balance

Substituting Eqs. 20.4-20.5 into Eq. 20.1:
$C[(T_{is} - T_{os})(w - w_s) + w_s(T_i - T_o)] + UA(T_v - T_o) =
mC\,dT_o/dt$. Subtract the steady state (Eq. 20.7) and transform:
$(mCs + UA + w_sC)T_o' = w_sCT_i' + UAT_v' - C(T_{os} - T_{is})W'$.
Divide by $UA + w_sC$: Eq. 20.9.

### Water flow to temperature in the closed kettle

With $G_2 = K_2/(\tau_ws + 1)$, $G_4 = 1/(\tau_vs + 1)$ and
$G_c = K_c$, $H = 1$, Eq. 20.21 reduces to the second-order Eq. 20.22
with $D_1 = 1 + K_5/R_v + K_cK_vK_2K_5 - K_2 > 0$ because $K_2 < 1$.

### Check -- the steady state is the linear model's fixed point

- Exact steady state against $T_{os} + K_2T_v'$ for a steam change

In [2]:
To = lambda w, Tv: (w * 60 + 750 * Tv) / (w + 750)
K2 = 750 / (750 + 250)
print(To(250, 260) - To(250, 250), K2 * 10)

7.5 7.5


**Code walkthrough.** Both print 7.5: Tv enters Eq. 20.1 linearly, so its gain K2 is exact
at fixed w. Only the products wTi and wTo needed approximation.

## Part B
### Absorber: Eliminating $X_1$

(Coughanowr & LeBlanc, Ch. 20, Sec. 20.2)
<!-- source: coughanowr-ch20 -->

From $sX_2 = cX_1 - aX_2$: $X_1 = (s + a)X_2/c$. Into the first:
$(s + a)^2X_2/c = bX_2 + cX_0$, so
$X_2/X_0 = c^2/[(s + a)^2 - bc]$. Divide by $a^2 - bc$: Eq. 20.34.
$\zeta^2 = a^2/(a^2 - bc)$; with $a = b + c$:
$\zeta^{-2} = 1 - \frac{bc}{(b + c)^2} = 1 - \frac{r}{(1 + r)^2}$,
$r = Vm/L$. Since $0 < r/(1 + r)^2 \le 1/4$ (maximum at $r = 1$),
$1 < \zeta \le 2/\sqrt3 = 1.155$: always overdamped, never far
from critical, and most damped at $Vm = L$.

### Check -- the range of the damping ratio

- Scan $r = Vm/L$

In [3]:
r = np.logspace(-2, 2, 2001)
zeta = 1 / np.sqrt(1 - r / (1 + r)**2)
print(round(zeta.min(), 4), round(zeta.max(), 4), r[zeta.argmax()])

1.0049 1.1547 1.0


**Code walkthrough.** Prints a minimum of 1.0049 (at the ends of the scan, r = 0.01 and
100) and a maximum of 1.1547 = 2/sqrt 3 at r = 1.0. The two-plate
absorber is overdamped for every flow ratio. The lecture's L/H = 1,
Vm/H = 1.5 give r = 1.5, r/(1 + r)^2 = 0.24 and zeta = 1.147.

## Part C
### Conduction into a Solid and the Exchanger

(Coughanowr & LeBlanc, Ch. 20, Sec. 20.3)
<!-- source: coughanowr-ch20 -->

### The slab (Eqs. 20.39-20.53)

$\alpha T_{xx} = T_t$; Laplace in $t$: $\alpha\bar T'' = s\bar T$, so
$\bar T = A_1e^{-\sqrt{s/\alpha}x}$ (bounded as $x \to \infty$).
$\bar T(L,s)/\bar T(0,s) = e^{-\sqrt{s/\alpha}L}$; step:
$T(L,t) = \mathrm{erfc}[L/\sqrt{4\alpha t}]$. With $s = j\omega$:
$AR = e^{-\sqrt{\omega/2\alpha}L}$, phase $-\sqrt{\omega/2\alpha}L$
-- unbounded.

### The exchanger (Eqs. 20.61-20.65)

$dT'/dx + \frac{s + 1/\tau}{v}T' = \frac{1}{\tau v}T_v'$ with
$T'(0,s)$ given: integrating factor $e^{(s + 1/\tau)x/v}$ gives
Eq. 20.63; at $x = L$, $e^{-(\tau s + 1)L/\tau v} = Ke^{-\tau_ds}$.

### Check -- the slab step response

- $\mathrm{erfc}$ at a few values of $\alpha t/L^2$

In [4]:
for z in [0.05, 0.1, 0.5, 1.0, 5.0]:
    print(f"alpha t / L^2 = {z}: T(L,t) = {erfc(0.5 / np.sqrt(z)):.3f}")

alpha t / L^2 = 0.05: T(L,t) = 0.002
alpha t / L^2 = 0.1: T(L,t) = 0.025
alpha t / L^2 = 0.5: T(L,t) = 0.317
alpha t / L^2 = 1.0: T(L,t) = 0.480
alpha t / L^2 = 5.0: T(L,t) = 0.752


**Code walkthrough.** Prints 0.002, 0.025, 0.317, 0.480, 0.752: Fig. 20-7. The response is
slow to finish (0.75 at a dimensionless time of 5) -- the long tail of
a distributed system.

### Check -- Ex. 20.1 numbers

- Velocity, tau, td, K

In [5]:
Ai = np.pi * (0.584 / 12)**2 / 4
v = 2 / 7.48 / Ai / 60
inv_tau = np.pi * (0.584 / 12) * 100 / (Ai * 62.4) / 3600
print(f"Ai = {Ai:.5f} ft2, v = {v:.2f} ft/s, tau = {1/inv_tau:.1f} s, "
      f"td = {40/v:.1f} s, K = {np.exp(-40/v*inv_tau):.3f}")

Ai = 0.00186 ft2, v = 2.40 ft/s, tau = 27.3 s, td = 16.7 s, K = 0.543


**Code walkthrough.** Prints Ai = 0.00186 ft2, v = 2.40 ft/s, tau = 27.3 s, td = 16.7 s and
K = 0.543. U is per hour, hence the /3600.